In [ ]:

from fastapi import (
    APIRouter,
    HTTPException,
    status,
    Depends,
)
from typing import Any

from app.api.auth import get_current_user
from app.utils.helpers import generate_id, utc_now, record_activity


router = APIRouter(
    prefix="/tutor",
    tags=["AI Tutor"],
)


def get_database():
    """Return the application's MongoDB wrapper."""

    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def verify_project_ownership(
    database,
    project_id: str,
    user_id: str,
):
    project = database.collection("projects").find_one(
        {
            "id": project_id,
            "user_id": user_id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return project


def clean_document(document: dict) -> dict:
    result = dict(document)
    result.pop("_id", None)
    return result


@router.post("/ask")
async def ask_tutor(
    request: dict[str, Any],
    current_user=Depends(get_current_user),
):
    """
    Ask the project-aware AI Tutor.

    The endpoint persists the conversation first and then attempts
    to use TutorService when the service exposes the expected API.
    """

    database = get_database()

    project_id = request.get("project_id")
    message = request.get("message")

    if not project_id:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="project_id is required.",
        )

    if not message or not str(message).strip():
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="message is required.",
        )

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    conversations = database.collection("conversations")

    conversation_id = request.get("conversation_id")

    conversation = None

    if conversation_id:
        conversation = conversations.find_one(
            {
                "id": conversation_id,
                "project_id": project_id,
                "user_id": current_user.id,
            }
        )

        if conversation is None:
            raise HTTPException(
                status_code=status.HTTP_404_NOT_FOUND,
                detail="Conversation not found.",
            )

    if conversation is None:
        conversation_id = generate_id()

        conversation = {
            "id": conversation_id,
            "project_id": project_id,
            "user_id": current_user.id,
            "title": str(message).strip()[:80],
            "messages": [],
            "created_at": utc_now(),
            "updated_at": utc_now(),
        }

        conversations.insert_one(conversation)

    user_message = {
        "role": "user",
        "content": str(message).strip(),
        "timestamp": utc_now(),
        "citations": [],
    }

    conversations.update_one(
        {"id": conversation_id},
        {
            "$push": {
                "messages": user_message,
            },
            "$set": {
                "updated_at": utc_now(),
            },
        },
    )

    assistant_content = None
    citations = []

    # Use only a small recent conversation window and concise mastery context.
    recent_messages = (conversation.get("messages") or [])[-8:]
    conversation_context = "\n".join(
        f"{item.get('role', 'unknown')}: {str(item.get('content', ''))[:1200]}"
        for item in recent_messages
    )

    mastery_documents = database.collection("mastery").find(
        {"user_id": current_user.id, "project_id": project_id},
        {"_id": 0, "concept_id": 1, "score": 1, "trend": 1},
    ).sort("score", 1).limit(5)
    learning_context = "\n".join(
        f"concept={item.get('concept_id')}, score={item.get('score')}, trend={item.get('trend', 'stable')}"
        for item in mastery_documents
    )

    try:
        from app.services.tutor_service import TutorService

        service = TutorService(database)
        result = service.answer(
            user_id=current_user.id,
            project_id=project_id,
            question=str(message).strip(),
            conversation=conversation_context,
            learning_context=learning_context,
            top_k=5,
        )

        assistant_content = getattr(result, "answer", None)
        citations = [
            {
                "material_id": item.material_id,
                "chunk_id": item.chunk_id,
                "page_number": item.page_number,
                "file_name": item.file_name,
            }
            for item in getattr(result, "citations", [])
        ]

    except Exception as exc:
        # Keep the existing fallback, but retain the real server-side failure.
        import logging
        logging.getLogger(__name__).exception("Tutor request failed")
        assistant_content = None

    if assistant_content is None:
        assistant_content = (
            "I could not generate an AI response right now. "
            "Your question has been saved to this conversation."
        )

    assistant_message = {
        "role": "assistant",
        "content": assistant_content,
        "timestamp": utc_now(),
        "citations": citations,
    }

    updated = conversations.find_one(
        {
            "id": conversation_id,
            "user_id": current_user.id,
            "project_id": project_id,
        }
    )

    record_activity(
        database,
        user_id=current_user.id,
        project_id=project_id,
        event_type="tutor_interaction",
        description="Tutor interaction",
        entity_type="conversation",
        entity_id=conversation_id,
        metadata={
            "message_length": len(str(message).strip()),
            "has_citations": bool(citations),
        },
    )

    return {
        "conversation_id": conversation_id,
        "message": assistant_message,
        "conversation": clean_document(updated),
    }


@router.get("/conversations/project/{project_id}")
async def list_conversations(
    project_id: str,
    current_user=Depends(get_current_user),
):
    """List conversations for an authorized project."""

    database = get_database()

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    conversations = database.collection("conversations")

    documents = conversations.find(
        {
            "project_id": project_id,
            "user_id": current_user.id,
        }
    ).sort(
        "updated_at",
        -1,
    )

    return [
        clean_document(document)
        for document in documents
    ]


@router.get("/conversations/{conversation_id}")
async def get_conversation(
    conversation_id: str,
    current_user=Depends(get_current_user),
):
    """Return one authorized conversation."""

    database = get_database()

    conversation = database.collection("conversations").find_one(
        {
            "id": conversation_id,
            "user_id": current_user.id,
        }
    )

    if conversation is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Conversation not found.",
        )

    return clean_document(conversation)
